# Sampling the spans to hide

T5 hides about 15% of a passage's tokens, in spans 3 tokens long on average. Before any
sentinel is written, something has to choose those spans. Here you write a sampler that
hides an exact number of tokens in an exact number of spans, at random places, with no
two spans touching.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. `runs` lists the lengths of the runs of `True` in a boolean array.
The checks use it, and you may too.

In [ ]:
import numpy as np


def runs(mask):
    """Lengths of the runs of True in a boolean array, from left to right."""
    lengths, current = [], 0
    for hidden in mask:
        if hidden:
            current += 1
        elif current:
            lengths.append(current)
            current = 0
    if current:
        lengths.append(current)
    return lengths


print(runs(np.array([0, 1, 1, 0, 0, 1, 0], dtype=bool)))   # two runs, of 2 and 1

## Part 1 (coding task): A random split

Write `random_split(total, k, rng)`. It returns a NumPy integer array of `k` positive
integers that add up to `total`, chosen so that every such list is equally likely. For
`total = 4` and `k = 2`, the three possible lists, `[1, 3]`, `[2, 2]` and `[3, 1]`,
should each come up a third of the time. Take all randomness from `rng`, a NumPy random
generator. You may assume `1 <= k <= total`.

In [ ]:
def random_split(total, k, rng):
    """k positive integers adding up to total, every such list equally likely."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
parts = random_split(10, 4, rng)
print(parts)
assert isinstance(parts, np.ndarray) and np.issubdtype(parts.dtype, np.integer), "return a NumPy array of integers"
assert len(parts) == 4, f"expected 4 parts, got {len(parts)}"
assert parts.sum() == 10, f"the parts should add up to 10, got {parts.sum()}"
assert (parts >= 1).all(), f"every part should be at least 1, got {parts}"
assert np.array_equal(random_split(5, 1, rng), [5]), "with k = 1 the only split is [total]"
assert np.array_equal(random_split(3, 3, rng), [1, 1, 1]), "with k = total every part is 1"
counts = {}
for _ in range(10000):
    key = tuple(int(p) for p in random_split(6, 3, rng))
    counts[key] = counts.get(key, 0) + 1
assert all(sum(key) == 6 and min(key) >= 1 for key in counts), "some split of 6 into 3 parts is wrong"
assert len(counts) == 10, f"6 splits into 3 positive parts in 10 ways, but {len(counts)} came up"
share = np.array(list(counts.values())) / 10000
assert np.abs(share - 0.1).max() < 0.02, (
    f"the splits are not equally likely: {dict(sorted(counts.items()))}. "
    "A hint: lay the total units in a row and cut it at k - 1 of the total - 1 gaps between neighbors."
)
print("Part 1 passed")

## Part 2 (coding task): The span mask

Write `span_mask(L, density, mean_span, rng)`, which returns a boolean array of length
`L`, True at the tokens to hide. It hides `n = round(L * density)` tokens in
`s = max(1, round(n / mean_span))` spans, using Python's `round`, and lays the passage
out as

kept run, span, kept run, span, ..., span, kept run

so the passage starts and ends with kept tokens and no two spans touch. The `s` span
lengths are a random split of `n`, and the `s + 1` kept runs a random split of `L - n`,
both from your `random_split`. For example, with `L = 12`, `n = 3` and `s = 1`, one
possible mask keeps 4 tokens, hides 3 and keeps 5. You may assume `L - n >= s + 1`.

In [ ]:
def span_mask(L, density, mean_span, rng):
    """A boolean mask of length L, True at the tokens to hide."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
m = span_mask(100, 0.15, 3, rng)
print(m.astype(int))
assert isinstance(m, np.ndarray) and m.dtype == bool and m.shape == (100,), "return a boolean array of length L"
for L, mean_span, n, s in [(100, 3, 15, 5), (37, 3, 6, 2), (20, 3, 3, 1), (200, 10, 30, 3)]:
    span_lengths, last_kept = set(), set()
    for _ in range(300):
        m = span_mask(L, 0.15, mean_span, rng)
        assert m.sum() == n, f"L = {L}: expected {n} hidden tokens, got {m.sum()}"
        assert len(runs(m)) == s, f"L = {L}: expected {s} spans, got {len(runs(m))} in {m.astype(int)}"
        assert not m[0] and not m[-1], f"L = {L}: the first and last tokens should be kept, got {m.astype(int)}"
        span_lengths.update(runs(m))
        last_kept.add(int(np.argmax(m[::-1])))   # kept tokens after the last span
    if s > 1:
        assert len(span_lengths) > 1, f"L = {L}: every span is {span_lengths.pop()} long: the span lengths should be a random split of n"
    assert len(last_kept) > 1, f"L = {L}: the last kept run is always {last_kept.pop()} long: it should come from the random split of L - n"
seen = np.zeros(40, dtype=int)
for _ in range(2000):
    seen += span_mask(40, 0.15, 3, rng)
never = np.flatnonzero(seen[1:-1] == 0) + 1
assert len(never) == 0, f"positions {never.tolist()} are never hidden"
print("Part 2 passed")

## Solution

### Part 1: A random split

In [ ]:
def random_split(total, k, rng):
    """k positive integers adding up to total, every such list equally likely."""
    cuts = np.sort(rng.choice(total - 1, size=k - 1, replace=False) + 1)  # gaps 1 .. total-1
    edges = np.concatenate(([0], cuts, [total]))
    return np.diff(edges)

**Why it works.** Number the gaps between neighboring units from 1 to `total - 1`.
Cutting at `k - 1` different gaps gives `k` pieces, each at least 1 long, and every split
comes from exactly one set of cuts. `rng.choice(total - 1, k - 1, replace=False)` picks
every set of gaps with the same probability, so every split is equally likely. Drawing
the parts one by one and fixing the total afterwards, or spreading the extra units with
`rng.multinomial`, gives the right sum but favors even splits: the multinomial version
gives `[2, 2, 2]` six times as often as `[1, 1, 4]`.

### Part 2: The span mask

In [ ]:
def span_mask(L, density, mean_span, rng):
    """A boolean mask of length L, True at the tokens to hide."""
    n = round(L * density)
    s = max(1, round(n / mean_span))
    spans = random_split(n, s, rng)
    kept = random_split(L - n, s + 1, rng)
    mask = []
    for i in range(s):
        mask += [False] * kept[i] + [True] * spans[i]
    mask += [False] * kept[s]
    return np.array(mask, dtype=bool)

**Why it works.** The counts are fixed first, so every mask hides exactly `n` tokens in
exactly `s` spans. Giving every kept run at least one token is what keeps two spans apart.
Two spans that touch would read as one longer span, and in the input they would become
two sentinels side by side with nothing between them. Laying out the `s + 1` kept runs
and the `s` spans in turn, starting and ending with a kept run, places everything, and
the two random splits decide where. With `L = 100`, the 85 kept tokens are cut into 6
runs and the 15 hidden ones into 5 spans of mean length 3.